In [1]:
import pandas as pd
import sys
import os
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi


In [2]:
humaneval = pd.read_csv("/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/humaneval_test_modified_open.csv", header = 0, encoding='unicode_escape')
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
sys.path.append(parent_dir)

In [3]:
from utility.humaneval_functions import HumanEvalHelper
from database import MongoDBHelper

In [4]:
sample_qn = humaneval.iloc[32]
prompt = sample_qn['prompt']
canonical_solution = sample_qn['canonical_solution']
original_test = sample_qn['test']
print(prompt)

import math


def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n
    """
    return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])


def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
    Moreover, find_zero only takes list xs having even number of coefficients
    and largest non zero coefficient as it guarantees
    a solution.
    >>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
    -0.5
    >>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
    1.0
    """



In [5]:
new_qn, qn_desc = HumanEvalHelper.seperate_original_desciptions(prompt)
print(new_qn)
print('-- qn_desc below --')
print(qn_desc)

import math

def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n
    """
    return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])

def find_zero(xs: list):
-- qn_desc below --
xs are coefficients of a polynomial.
find_zero find x such that poly(x) = 0.
find_zero returns only only zero point, even if there are many.
Moreover, find_zero only takes list xs having even number of coefficients
and largest non zero coefficient as it guarantees
a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0


In [6]:
desc, examples = HumanEvalHelper.extract_examples(qn_desc)

In [7]:
check_function = HumanEvalHelper.process_original_tests(original_test)

In [8]:
complete_solution = new_qn + "\n" + canonical_solution
check = HumanEvalHelper.check_test_case(test_case=check_function, code_snippet = complete_solution)

print("Function has passed the test" if check else "Function did not pass the test")

Function has passed the test


In [9]:
mongodbHelper = MongoDBHelper()
mongodbHelper.check_database_connectivity()

True

In [10]:
db = mongodbHelper.client["Base_Questions_DB"]
open_ended_db = db["HumanEval_Open_Ended"]

In [11]:
to_check_example = []
to_check_test = set()

for idx in range(humaneval.__len__()):

    qn_id = f"HumanEvalo{idx-len(to_check_example)-len(to_check_test)}"

    qn_details = open_ended_db.find_one({"_id" : qn_id})        # checking if this qn_id already exists in the db

    qn = humaneval.iloc[idx]
    prompt = qn['prompt']
    canonical_solution = qn['canonical_solution']
    tests = qn['test']
    original_test_id = qn['task_id']

    new_qn, qn_desc = HumanEvalHelper.seperate_original_desciptions(prompt)

    desc, examples = HumanEvalHelper.extract_examples(qn_desc)

    if len(examples) < 1:
        to_check_example.append(qn_id)

        continue

    check_function = HumanEvalHelper.process_original_tests(tests)

    full_solution = new_qn + "\n" + canonical_solution

    code_validation = HumanEvalHelper.check_test_case(test_case = check_function, code_snippet = full_solution)
    
    entry_dict = {
        "_id" : qn_id,
        "qn" : new_qn,
        "canon_solution" : canonical_solution,
        "qn_desc" : desc,
        "examples": examples,
        "check" : check_function,
        "original_id": original_test_id
    }
    if code_validation:
        if qn_details is None:
            open_ended_db.insert_one(entry_dict)
            print('Added entry to database: {id}'.format(id = qn_id))
        else:
            open_ended_db.update_one({"_id" : qn_id}, update = {"$set": entry_dict})
            print('Updated existing entry in database: {id}'.format(id = qn_id))
    else:
        to_check_test.add(qn_id)

print(to_check_example if len(to_check_example) > 0 else "All cases contains examples. Nothing to check!")
print(to_check_test if len(to_check_test) > 0 else "All test cases passed. Nothing to check!")

Added entry to database: HumanEvalo0
Added entry to database: HumanEvalo1
Added entry to database: HumanEvalo2
Added entry to database: HumanEvalo3
Added entry to database: HumanEvalo4
Added entry to database: HumanEvalo5
Added entry to database: HumanEvalo6
Added entry to database: HumanEvalo7
Added entry to database: HumanEvalo8
Added entry to database: HumanEvalo9
Added entry to database: HumanEvalo10
Added entry to database: HumanEvalo11
Added entry to database: HumanEvalo12
Added entry to database: HumanEvalo13
Added entry to database: HumanEvalo14
Added entry to database: HumanEvalo15
Added entry to database: HumanEvalo16
Added entry to database: HumanEvalo17
Added entry to database: HumanEvalo18
Added entry to database: HumanEvalo19
Added entry to database: HumanEvalo20
Added entry to database: HumanEvalo21
Added entry to database: HumanEvalo22
Added entry to database: HumanEvalo23
Added entry to database: HumanEvalo24
Added entry to database: HumanEvalo25
Added entry to databas

Modified question HumanEval/10 to contain nested functions for uniformity.
HumanEval/41, HumanEval/38, HumanEval/50 did not have any examples in the quetions. 
HumanEval/66 to HumanEval/163 modified to fit the standard doctest format.